In [1]:
import pandas as pd
import numpy as np
import math

In [23]:
warehouses = pd.DataFrame({
    'warehouse_id': ['W1','W2','W3','W4','W5','W6','W7','W8'],
    'city': ['Casablanca','Rabat','Fès','Marrakech',
             'Agadir','Tanger','Oujda','Laâyoune'],
    'latitude': [33.5731, 34.0209, 34.0331, 31.6295,
                 30.4278, 35.7595, 34.6814, 27.1418],
    'longitude': [-7.5898, -6.8416, -5.0003, -7.9811,
                  -9.5981, -5.8340, -1.9086, -13.1800],
    'capacity_tonnes': [12000, 8000, 7000, 9000,
                        6000, 7500, 5000, 4000],
    'monthly_cost_mad': [180000, 120000, 100000, 130000,
                         90000, 110000, 80000, 70000]
})

stores = pd.DataFrame({
    'store_id': [f'S{str(i).zfill(3)}' for i in range(1, 21)],
    'city': ['Casablanca','Rabat','Fès','Marrakech','Agadir',
             'Tanger','Oujda','Laâyoune','Meknès','Béni Mellal',
             'Kénitra','Tétouan','Safi','El Jadida','Nador',
             'Settat','Khouribga','Essaouira','Ouarzazate','Dakhla'],
    'region': ['Centre','Centre','Nord','Sud','Sud',
               'Nord','Est','Sud','Centre','Centre',
               'Nord','Nord','Ouest','Ouest','Est',
               'Centre','Centre','Ouest','Sud','Sud'],
    'latitude': [33.5731, 34.0209, 34.0331, 31.6295, 30.4278,
                 35.7595, 34.6814, 27.1418, 33.8935, 32.3372,
                 34.2610, 35.5785, 32.2994, 33.2316, 35.1740,
                 33.0014, 32.8811, 31.5125, 30.9335, 23.7136],
    'longitude': [-7.5898, -6.8416, -5.0003, -7.9811, -9.5981,
                  -5.8340, -1.9086, -13.1800, -5.5473, -6.3498,
                  -6.5802, -5.3684, -9.2372, -8.5007, -2.9287,
                  -7.6166, -6.9063, -9.7749, -6.9370, -15.9355],
    'monthly_demand_tonnes': [4500, 3200, 2800, 3100, 1900,
                               2200, 1500, 800, 2000, 1700,
                               1800, 1600, 1400, 1500, 1200,
                               1300, 1100, 900, 850, 600]
})
warehouses.set_index('warehouse_id', inplace=True)
stores.set_index('store_id', inplace=True)


In [24]:
warehouses.to_csv('../data/raw/warehouses.csv', index=False)
stores.to_csv('../data/raw/stores.csv', index=False)

In [30]:
from geopy import distance

distances = []
def get_cord(row):
    return (row['latitude'],row['longitude'])
for _,wh in warehouses.iterrows():
    for _,st in stores.iterrows():
        dist = distance.geodesic(get_cord(wh),get_cord(st)).km
        distances.append({
            'warehouse_id': wh.name,
            'store_id': st.name,
            'distance_km': dist
        })

distances_df = pd.DataFrame(distances)
distances_df = distances_df.merge(
    stores[['monthly_demand_tonnes']],
    left_on='store_id',
    right_index=True
)

# Then calculate realistic transport cost
# Transport cost — update in your distance matrix calculation
distances_df['transport_cost_mad'] = (
    distances_df['distance_km'] *
    distances_df['monthly_demand_tonnes'] *
    0.35
)

distances_df['delivery_days'] = np.ceil(distances_df['distance_km'] / 500).astype(int)
distances_df['is_feasible'] = distances_df['delivery_days'] <=2
distances_df.to_csv('../data/raw/distance_matrix.csv', index=False)
        

In [31]:
most_covering_warehouses = (
    distances_df[distances_df['is_feasible']]
    .groupby('warehouse_id').
    size().
    sort_values(ascending=False)
)
print(most_covering_warehouses)


warehouse_id
W5    20
W1    19
W4    19
W2    19
W3    18
W6    18
W7    18
W8    13
dtype: int64


W5 (Agadir) covers all 20 stores within 2 days, making it the most geographically central warehouse in the network. W8 (Laâyoune) covers only 13 stores  the most limited warehouse due to its remote southern location

In [32]:
stores_distances = (
    distances_df[distances_df['distance_km'] > 0]
    .groupby('store_id')['distance_km']
    .min()
    .sort_values(ascending=False)
)
print(stores_distances)

store_id
S008    504.857327
S020    470.133707
S007    293.361411
S006    206.124777
S005    203.919408
S004    203.919408
S010    172.993942
S003    170.057359
S013    140.058075
S019    125.854525
S018    121.439829
S015    108.043620
S017     99.746388
S014     92.817346
S001     85.249993
S002     85.249993
S016     63.456132
S009     52.874336
S012     46.694083
S011     35.924681
Name: distance_km, dtype: float64


S008 (Oujda) is the hardest store to serve in the network  its nearest warehouse is 504 km away. S020 (Dakhla) is second at 470 km, reflecting the challenge of serving Morocco's deep south. S011 (Béni Mellal) is the easiest non-same-city store to serve at only 36 km from its nearest warehouse.

In [38]:
cheapest_idx = (
    distances_df[distances_df['distance_km'] > 0].groupby('store_id')['distance_km']
    .idxmin()
)
cheapest_routes = distances_df.loc[cheapest_idx]
total_cost = cheapest_routes['transport_cost_mad'].sum()
monthly_transport = total_cost
print(f"Transport cost: {monthly_transport:,.0f} MAD/month")

monthly_warehouse = warehouses['monthly_cost_mad'].sum()
print(f"Warehouse operating cost: {monthly_warehouse:,.0f} MAD/month")

holding_rate = 0.03       

monthly_demand_total = stores['monthly_demand_tonnes'].sum()
monthly_holding = monthly_demand_total * holding_rate
print(f"Holding cost: {monthly_holding:,.0f} MAD/month")

total_monthly = monthly_transport + monthly_warehouse + monthly_holding
total_annual = total_monthly * 12
print(f"\nTotal monthly baseline cost: {total_monthly:,.0f} MAD/month")
print(f"Total annual baseline cost: {total_annual:,.0f} MAD/year")

Transport cost: 1,800,514 MAD/month
Warehouse operating cost: 880,000 MAD/month
Holding cost: 1,078 MAD/month

Total monthly baseline cost: 2,681,592 MAD/month
Total annual baseline cost: 32,179,105 MAD/year


## Baseline Cost Summary

| Cost Component | Monthly (MAD) | Annual (MAD) |
|---|---|---|
| Transport | 1,800,514 | 21,606,168 |
| Warehouse Operations | 880,000 | 10,560,000 |
| Holding & Spoilage | 494,400 | 5,932,800 |
| **Total Baseline** | **3,174,914** | **38,098,968** |

